<a href="https://colab.research.google.com/github/AlperYildirim1/HAMON/blob/main/HAMON_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
HAMON: Hardware-Accelerated Modulation Optical Network
=======================================================
A pure diffractive optical neural network (D²NN) for time series forecasting.

All computation is performed by light diffracting through free space and
passing through trainable phase masks. Zero digital linear layers.

Physics:
  - Input: SEQ_LEN steps encoded as light amplitude + PRED_LEN steps of darkness
  - N layers of trainable Phase Masks separated by free space (Angular Spectrum Method)
  - Detector reads coherent amplitude from the dark zone → forecast

This code is combined version of separated code cells
"""

# =============================================================================
# 0. IMPORTS & ENVIRONMENT
# =============================================================================

import os
import sys
import math
import json
import random
import logging
import argparse
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# --- Google Drive (Colab) ---

BASE_DIR = "/content/drive/MyDrive/HAMON_720"
SAVE_DIR = os.path.join(BASE_DIR, "checkpoints")
LOG_DIR = os.path.join(BASE_DIR, "logs")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Save dir: {SAVE_DIR}")
print(f"Checkpoints: {os.listdir(SAVE_DIR)}")

# --- Datasets ---
# Update paths to match your data location
# =============================================================================
# DATASET DOWNLOAD (HuggingFace)
# =============================================================================
import os, requests
from pathlib import Path

HF_BASE = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/"

DATASETS_CONFIG = {
    "etth1":    {"path": "ETT-small/ETTh1.csv",             "split": [8640, 2880, 2880],     "channels": 7,   "batch_size": 128},
    "etth2":    {"path": "ETT-small/ETTh2.csv",             "split": [8640, 2880, 2880],     "channels": 7,   "batch_size": 128},
    "ettm1":    {"path": "ETT-small/ETTm1.csv",             "split": [34560, 11520, 11520],  "channels": 7,   "batch_size": 128},
    "ettm2":    {"path": "ETT-small/ETTm2.csv",             "split": [34560, 11520, 11520],  "channels": 7,   "batch_size": 128},
    "weather":  {"path": "weather/weather.csv",              "split": "70_10_20",             "channels": 21,  "batch_size": 64},
    "exchange": {"path": "exchange_rate/exchange_rate.csv",  "split": "70_10_20",             "channels": 8,   "batch_size": 128},
    #"electricity": {"path": "electricity/electricity.csv", "split": "70_10_20", "channels": 321, "batch_size": 64,},
    #"traffic":     {"path": "traffic/traffic.csv",         "split": "70_10_20", "channels": 862, "batch_size": 16,},
}



for name, cfg in DATASETS_CONFIG.items():
    local_path = cfg["path"]
    if os.path.exists(local_path):
        print(f"  {name}: OK")
        continue

    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    url = HF_BASE + local_path
    print(f"  Downloading {name}... ", end="")
    r = requests.get(url)
    if r.status_code == 200:
        with open(local_path, "wb") as f:
            f.write(r.content)
        print("OK")
    else:
        print(f"FAILED ({r.status_code})")

print("\nAll datasets ready.")

# --- Training Hyperparameters ---
SEQ_LEN = 720
PRED_LENS = [96, 192, 336, 720]
LAYER_CANDIDATES = [16]

EPOCHS = 80
LR = 1e-2
WEIGHT_DECAY = 1e-3
PATIENCE = 15
GRAD_CLIP = 1.0
BF_LAMBDA = 0.5  # Backcast + Forecast joint supervision weight

# --- Optical Hardware Parameters ---
GRID_SIZE = 2048     # Pixels on the optical grid
SPACING = 10e-6      # 10 microns per pixel
WAVELENGTH = 1e-6    # 1 micron (near-infrared)
DISTANCE = 0.03      # 3 cm air gap between masks

# --- Seeds for reproducibility ---
SEEDS = [1, 2, 3]

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================

def set_seed(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# 3. LOGGING
# =============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


def get_logger(run_name):
    """Create a logger that writes to both file and console."""
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    log_path = os.path.join(LOG_DIR, f"{run_name}_{timestamp}.log")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(logging.Formatter(
        "%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    ))
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)

    return logger


# =============================================================================
# 4. DATA LOADING
# =============================================================================

class TSDataset(Dataset):
    """Simple sliding-window time series dataset."""

    def __init__(self, data, seq_len, pred_len):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_len]
        y = self.data[idx + self.seq_len: idx + self.seq_len + self.pred_len]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def load_data(dataset_name):
    cfg = DATASETS_CONFIG[dataset_name]
    df = pd.read_csv(cfg["path"])
    data = df.iloc[:, 1:].values

    if cfg["split"] == "70_10_20":
        n = len(data)
        train_len = int(n * 0.7)
        val_len = int(n * 0.1)
    else:
        train_len = cfg["split"][0]
        val_len = cfg["split"][1]

    scaler = StandardScaler()
    scaler.fit(data[:train_len])
    data = scaler.transform(data)

    train = data[:train_len]
    val = data[train_len: train_len + val_len]
    test = data[train_len + val_len:]
    return train, val, test, scaler, cfg


def make_loaders(train_data, val_data, test_data, pred_len, batch_size):
    """Create DataLoaders for train/val/test."""
    train_loader = DataLoader(
        TSDataset(train_data, SEQ_LEN, pred_len),
        batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=True,
    )
    val_loader = DataLoader(
        TSDataset(val_data, SEQ_LEN, pred_len),
        batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True,
    )
    test_loader = DataLoader(
        TSDataset(test_data, SEQ_LEN, pred_len),
        batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True,
    )
    return train_loader, val_loader, test_loader


# =============================================================================
# 5. MODEL: HAMON (Diffractive Optical Network)
# =============================================================================

class RevIN(nn.Module):
    """Reversible Instance Normalization (Kim et al., 2022)."""

    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x, mode="norm"):
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self._mean) / self._std
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / self.affine_weight
            return x * self._std + self._mean


class HAMON(nn.Module):
    """
    Hardware-Accelerated Modulation Optical Network.

    Simulates light propagating through N trainable phase masks
    separated by free-space gaps.

    Physics engine: 1D Angular Spectrum Method (rigorous diffraction).

    Args:
        lookback:    Input sequence length (default: 336)
        horizon:     Forecast length (default: 96)
        channels:    Number of variates (default: 7)
        num_layers:  Number of phase masks / glass plates (default: 4)
        grid_size:   Width of optical grid in pixels (default: 512)
        spacing:     Physical distance between pixels in meters (default: 10e-6)
        wavelength:  Light wavelength in meters (default: 1e-6)
        distance:    Air gap between masks in meters (default: 0.03)
    """

    def __init__(
        self,
        lookback=720,
        horizon=96,
        channels=7,
        num_layers=4,
        grid_size=GRID_SIZE,
        spacing=SPACING,
        wavelength=WAVELENGTH,
        distance=DISTANCE,
    ):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.channels = channels
        self.grid_size = grid_size
        self.num_layers = num_layers

        # Center data in the optical grid
        total_len = lookback + horizon
        assert total_len <= grid_size, (
            f"grid_size ({grid_size}) must be >= lookback + horizon ({total_len})"
        )
        self.start_idx = (grid_size - total_len) // 2

        # Digital pre/post-processing
        self.revin = RevIN(channels)

        # Trainable Phase Masks — the ONLY learned optical parameters
        self.phase_masks = nn.ParameterList([
            nn.Parameter(torch.zeros(grid_size)) for _ in range(num_layers)
        ])

        # Fixed free-space transfer function H (Angular Spectrum Method)
        fx = torch.fft.fftfreq(grid_size, d=spacing)
        k = 2 * math.pi / wavelength

        # Evanescent wave mask: frequencies beyond diffraction limit
        evanescent_mask = (wavelength * fx) ** 2 <= 1.0

        # Phase accumulated by each spatial frequency over distance z
        phase_shift = k * distance * torch.sqrt(
            torch.clamp(1.0 - (wavelength * fx) ** 2, min=0.0)
        )

        # Complex transfer function (fixed, not trainable)
        H = torch.exp(1j * phase_shift) * evanescent_mask
        self.register_buffer("H", H)

    def _propagate(self, U):
        """Free-space propagation via Angular Spectrum Method."""
        U_f = torch.fft.fft(U, norm="ortho")
        U_f = U_f * self.H
        return torch.fft.ifft(U_f, norm="ortho")

    def forward(self, x, return_backcast=False):
        B, L, C = x.shape

        # Digital: Normalize
        x_norm = self.revin(x, mode="norm")
        x_flat = x_norm.permute(0, 2, 1).reshape(B * C, L)

        # Electro-Optical: Encode data as light amplitude
        U_in = torch.zeros(
            B * C, self.grid_size, dtype=torch.complex64, device=x.device
        )
        U_in[:, self.start_idx: self.start_idx + L] = x_flat.to(torch.complex64)

        # ====================================================================
        #  THE TRANSFER MATRIX TRICK
        # Optik çekirdek tamamen lineer olduğu için, devasa (B*C, G) tensörünü
        # katman katman propagate etmek yerine, GxG bir Birim Matrisi (Identity)
        # bir kez geçirip sistemin "Geçiş Matrisini (M)" elde ediyoruz.
        # ====================================================================

        # 1. GxG Birim Matrisi oluştur (1120 x 1120)
        I = torch.eye(self.grid_size, dtype=torch.complex64, device=x.device)

        # Fazları önceden e^jX formatına çevir (her döngüde tekrar hesaplamamak için)
        complex_masks = [torch.exp(1j * mask) for mask in self.phase_masks]

        # 2. SADECE Birim Matrisi (I) optik çekirdekten geçir
        M = I
        for complex_mask in complex_masks:
            M = self._propagate(M)                   # Air gap (diffraction)
            M = M * complex_mask                     # Glass plate (phase shift)

        M = self._propagate(M) # Final dedektör boşluğu

        # 3. Devasa verimizi bu matrisle TEK SEFERDE çarp! (cuBLAS hızı)
        U = torch.matmul(U_in, M)
        # ====================================================================

        # Opto-Electrical: Coherent detection (read real part)
        amplitude = torch.real(U)

        # Read detector regions
        bc_start = self.start_idx
        fc_start = self.start_idx + self.lookback
        backcast_flat = amplitude[:, bc_start: bc_start + self.lookback]
        forecast_flat = amplitude[:, fc_start: fc_start + self.horizon]

        # Digital: Reshape and denormalize
        backcast = backcast_flat.reshape(B, C, self.lookback).permute(0, 2, 1)
        forecast = forecast_flat.reshape(B, C, self.horizon).permute(0, 2, 1)
        forecast = self.revin(forecast, mode="denorm")

        if return_backcast:
            return forecast, backcast, x_norm
        return forecast

    def optical_summary(self):
        """Print a summary of the optical system."""
        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        n_optical = sum(p.numel() for p in self.phase_masks)
        device_len = (self.num_layers + 1) * DISTANCE * 100  # cm
        return {
            "total_params": n_params,
            "optical_params": n_optical,
            "num_layers": self.num_layers,
            "device_length_cm": device_len,
            "grid_size": self.grid_size,
            "spacing_um": SPACING * 1e6,
            "wavelength_um": WAVELENGTH * 1e6,
            "distance_cm": DISTANCE * 100,
        }


# =============================================================================
# 6. TRAINING & EVALUATION
# =============================================================================

def evaluate(model, loader, criterion):
    """Evaluate model on a DataLoader, returning MSE and MAE."""
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_mse += criterion(pred, y).item() * x.size(0)
            total_mae += nn.functional.l1_loss(pred, y).item() * x.size(0)
            n += x.size(0)
    return total_mse / n, total_mae / n


def train_single(
    model, train_loader, val_loader, ckpt_path, logger,
    epochs=EPOCHS, patience=PATIENCE
):
    """Train a single model configuration. Returns best validation MSE."""
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    best_val_mse = float("inf")
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            forecast, backcast, x_norm = model(x, return_backcast=True)
            loss_fc = criterion(forecast, y)
            loss_bc = criterion(backcast, x_norm)
            loss = loss_fc + BF_LAMBDA * loss_bc

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            train_losses.append(loss_fc.item())

        scheduler.step()
        avg_train = sum(train_losses) / len(train_losses)
        val_mse, val_mae = evaluate(model, val_loader, criterion)

        logger.info(
            f"  Epoch {epoch:3d}/{epochs} | "
            f"Train: {avg_train:.6f} | "
            f"Val MSE: {val_mse:.6f} MAE: {val_mae:.6f} | "
            f"LR: {scheduler.get_last_lr()[0]:.2e}"
        )

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logger.info(f"  Early stopping at epoch {epoch}")
                break

    return best_val_mse


# =============================================================================
# 7. MAIN BENCHMARK LOOP
# =============================================================================

def run_benchmark(
    datasets=None,
    pred_lens=None,
    layer_candidates=None,
    seeds=None,
):
    """
    Run the full HAMON benchmark.

    For each (dataset, horizon):
      1. Try all layer candidates, pick best on validation
      2. Report test MSE/MAE averaged over seeds
    """
    results_path = os.path.join(RESULTS_DIR, "hamon_results.json")
    datasets = datasets or list(DATASETS_CONFIG.keys())
    pred_lens = pred_lens or PRED_LENS
    layer_candidates = layer_candidates or LAYER_CANDIDATES
    seeds = seeds or SEEDS

    all_results = []
    master_logger = get_logger("HAMON_benchmark")

    master_logger.info("=" * 70)
    master_logger.info("HAMON BENCHMARK")
    master_logger.info(f"Datasets: {datasets}")
    master_logger.info(f"Horizons: {pred_lens}")
    master_logger.info(f"Layer candidates: {layer_candidates}")
    master_logger.info(f"Seeds: {seeds}")
    master_logger.info(f"Device: {device}")
    master_logger.info(f"Save dir: {SAVE_DIR}")
    master_logger.info("=" * 70)

    for dataset_name in datasets:
        # --- Load data ---
        try:
            train_data, val_data, test_data, scaler, cfg = load_data(dataset_name)
        except FileNotFoundError as e:
            master_logger.info(f"SKIP {dataset_name}: {e}")
            continue

        master_logger.info(f"\n{'='*70}")
        master_logger.info(f"DATASET: {dataset_name} | Channels: {cfg['channels']}")
        master_logger.info(f"{'='*70}")

        for pred_len in pred_lens:
            master_logger.info(f"\n  --- Horizon: {pred_len} ---")

            train_loader, val_loader, test_loader = make_loaders(
                train_data, val_data, test_data,
                pred_len, cfg["batch_size"]
            )

            # --- Phase 1: Select best num_layers on validation (seed=1) ---
            set_seed(SEEDS[0])
            best_layers = None
            best_val = float("inf")

            for num_layers in layer_candidates:
                run_name = f"HAMON_{dataset_name}_h{pred_len}_L{num_layers}_select"
                logger = get_logger(run_name)

                model = HAMON(
                    lookback=SEQ_LEN, horizon=pred_len,
                    channels=cfg["channels"], num_layers=num_layers,
                ).to(device)

                summary = model.optical_summary()
                logger.info(
                    f"  Trying L={num_layers} | "
                    f"Params: {summary['total_params']:,} | "
                    f"Device: {summary['device_length_cm']:.0f}cm"
                )

                ckpt = os.path.join(SAVE_DIR, f"{run_name}.pt")
                val_mse = train_single(
                    model, train_loader, val_loader, ckpt, logger
                )

                master_logger.info(
                    f"    L={num_layers:2d} | Val MSE: {val_mse:.6f}"
                )

                if val_mse < best_val:
                    best_val = val_mse
                    best_layers = num_layers

            master_logger.info(
                f"  >> Best layers: {best_layers} (Val MSE: {best_val:.6f})"
            )

            # --- Phase 2: Train with best layers across all seeds ---
            seed_mses, seed_maes = [], []

            for seed in seeds:
                set_seed(seed)
                run_name = f"HAMON_{dataset_name}_h{pred_len}_L{best_layers}_s{seed}"
                logger = get_logger(run_name)
                logger.info(f"  Seed {seed} | L={best_layers}")

                model = HAMON(
                    lookback=SEQ_LEN, horizon=pred_len,
                    channels=cfg["channels"], num_layers=best_layers,
                ).to(device)

                ckpt = os.path.join(SAVE_DIR, f"{run_name}.pt")
                train_single(model, train_loader, val_loader, ckpt, logger)

                # Test
                model.load_state_dict(
                    torch.load(ckpt, weights_only=True, map_location=device)
                )
                test_mse, test_mae = evaluate(
                    model, test_loader, nn.MSELoss()
                )
                seed_mses.append(test_mse)
                seed_maes.append(test_mae)

                logger.info(
                    f"    Seed {seed} TEST | MSE: {test_mse:.6f} MAE: {test_mae:.6f}"
                )

            # --- Aggregate ---
            avg_mse = np.mean(seed_mses)
            std_mse = np.std(seed_mses)
            avg_mae = np.mean(seed_maes)
            std_mae = np.std(seed_maes)

            summary = HAMON(
                lookback=SEQ_LEN, horizon=pred_len,
                channels=cfg["channels"], num_layers=best_layers,
            ).optical_summary()

            result = {
                "dataset": dataset_name,
                "horizon": pred_len,
                "layers": best_layers,
                "params": summary["total_params"],
                "optical_params": summary["optical_params"],
                "device_cm": summary["device_length_cm"],
                "mse_mean": float(avg_mse),
                "mse_std": float(std_mse),
                "mae_mean": float(avg_mae),
                "mae_std": float(std_mae),
                "mse_per_seed": [float(m) for m in seed_mses],
                "mae_per_seed": [float(m) for m in seed_maes],
            }
            all_results.append(result)

            master_logger.info(
                f"  RESULT | {dataset_name} H={pred_len} L={best_layers} | "
                f"MSE: {avg_mse:.6f}±{std_mse:.6f} | "
                f"MAE: {avg_mae:.6f}±{std_mae:.6f} | "
                f"Params: {summary['total_params']:,}"
            )

            # --- Save intermediate results ---
            results_path = os.path.join(RESULTS_DIR, "hamon_results.json")
            with open(results_path, "w") as f:
                json.dump(all_results, f, indent=2)

    # --- Final Summary ---
    master_logger.info(f"\n\n{'='*80}")
    master_logger.info("HAMON — FULL BENCHMARK RESULTS")
    master_logger.info(f"{'='*80}")
    master_logger.info(
        f"{'Dataset':<12} {'H':>4} {'L':>3} {'Params':>7} "
        f"{'MSE':>12} {'MAE':>12}"
    )
    master_logger.info("-" * 80)

    for r in all_results:
        master_logger.info(
            f"{r['dataset']:<12} {r['horizon']:>4d} {r['layers']:>3d} "
            f"{r['params']:>7d} "
            f"{r['mse_mean']:>7.4f}±{r['mse_std']:.4f} "
            f"{r['mae_mean']:>7.4f}±{r['mae_std']:.4f}"
        )

    master_logger.info(f"{'='*80}")
    master_logger.info(f"Results saved to {results_path}")

    return all_results


# =============================================================================
# 8. ENTRY POINT
# =============================================================================

results = run_benchmark(
    datasets=["etth1", "etth2", "ettm1", "ettm2", "weather", "exchange"],
    pred_lens=[96, 192, 336, 720],
    layer_candidates=[16],
    seeds=[1, 2, 3],
)

#results = run_benchmark(
#    datasets=["electricity", "traffic"],
#    pred_lens=[96, 192, 336, 720],
#    layer_candidates=[16],
#    seeds=[1, 2, 3],
#)

Save dir: /content/drive/MyDrive/HAMON_720/checkpoints
Checkpoints: []
  etth1: OK
  etth2: OK
  ettm1: OK
  ettm2: OK
  weather: OK
  exchange: OK

All datasets ready.


HAMON BENCHMARK
Datasets: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'exchange']
Horizons: [96, 192, 336, 720]
Layer candidates: [16]
Seeds: [1, 2, 3]
Device: cuda
Save dir: /content/drive/MyDrive/HAMON_720/checkpoints

DATASET: etth1 | Channels: 7

  --- Horizon: 96 ---


Device: cuda


  Trying L=16 | Params: 32,782 | Device: 51cm
  Epoch   1/80 | Train: 0.512354 | Val MSE: 0.740667 MAE: 0.608340 | LR: 1.00e-02
  Epoch   2/80 | Train: 0.385090 | Val MSE: 0.730996 MAE: 0.591823 | LR: 9.98e-03
  Epoch   3/80 | Train: 0.363268 | Val MSE: 0.719750 MAE: 0.582529 | LR: 9.97e-03
  Epoch   4/80 | Train: 0.352837 | Val MSE: 0.714948 MAE: 0.577843 | LR: 9.94e-03
  Epoch   5/80 | Train: 0.347151 | Val MSE: 0.701503 MAE: 0.571585 | LR: 9.90e-03
  Epoch   6/80 | Train: 0.344482 | Val MSE: 0.717375 MAE: 0.575248 | LR: 9.86e-03
  Epoch   7/80 | Train: 0.342429 | Val MSE: 0.715278 MAE: 0.573407 | LR: 9.81e-03
  Epoch   8/80 | Train: 0.341040 | Val MSE: 0.712429 MAE: 0.573087 | LR: 9.76e-03
  Epoch   9/80 | Train: 0.339701 | Val MSE: 0.707150 MAE: 0.571101 | LR: 9.69e-03
  Epoch  10/80 | Train: 0.339223 | Val MSE: 0.715906 MAE: 0.574307 | LR: 9.62e-03
  Epoch  11/80 | Train: 0.339006 | Val MSE: 0.708530 MAE: 0.572144 | LR: 9.54e-03
  Epoch  12/80 | Train: 0.338108 | Val MSE: 0.712201

In [ ]:
import os
import json
import pandas as pd
from google.colab import drive


RESULTS_PATH = "/content/drive/MyDrive/HAMON/results/hamon_results.json"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)

    df = pd.DataFrame(results)

    df['MSE (Mean ± Std)'] = df.apply(lambda row: f"{row['mse_mean']:.4f} ± {row['mse_std']:.4f}", axis=1)
    df['MAE (Mean ± Std)'] = df.apply(lambda row: f"{row['mae_mean']:.4f} ± {row['mae_std']:.4f}", axis=1)


    display_df = df[['dataset', 'horizon', 'layers', 'params', 'MSE (Mean ± Std)', 'MAE (Mean ± Std)']]
    display_df.columns = ['Dataset', 'Horizon', 'Layers', 'Params', 'MSE', 'MAE']


    print(display_df.to_string(index=False))


                         HAMON BENCHMARK SONUÇLARI
 Dataset  Horizon  Layers  Params             MSE             MAE
   etth1       96      16   17934 0.4440 ± 0.0008 0.4400 ± 0.0008
   etth1      192      16   17934 0.4968 ± 0.0024 0.4769 ± 0.0017
   etth1      336      16   17934 0.5486 ± 0.0031 0.5117 ± 0.0014
   etth1      720      16   17934 0.7064 ± 0.0062 0.6099 ± 0.0034
   etth2       96      16   17934 0.2343 ± 0.0025 0.3251 ± 0.0024
   etth2      192      16   17934 0.2937 ± 0.0019 0.3713 ± 0.0020
   etth2      336      16   17934 0.3411 ± 0.0021 0.4039 ± 0.0013
   etth2      720      16   17934 0.4698 ± 0.0037 0.4818 ± 0.0022
   ettm1       96      16   17934 0.3724 ± 0.0003 0.3881 ± 0.0007
   ettm1      192      16   17934 0.4186 ± 0.0016 0.4137 ± 0.0011
   ettm1      336      16   17934 0.4703 ± 0.0010 0.4451 ± 0.0016
   ettm1      720      16   17934 0.5530 ± 0.0011 0.4965 ± 0.0029
   ettm2       96      16   17934 0.1484 ± 0.0004 0.2514 ± 0.0003
   ettm2      192      16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')